In [23]:
#6.3
from matplotlib import Axes
from Axes import imshow
imshow(m101.fits)

ImportError: cannot import name 'Axes' from 'matplotlib' (/Software/users/modules/9/software/anaconda3/2024.02/lib/python3.11/site-packages/matplotlib/__init__.py)

In [4]:
#6.6 
from astropy.io import fits
import numpy as np
from math import log

# Open the FITS file
hdul = fits.open('APOGEE_stars.fits')

# Access the primary data (in this case, it is a Table)
data = hdul[1].data  # Often, the first extension contains the table
colnames = hdul[1].columns.names # List of the column names

print(f"The difference between reading an image vs reading a table using Astropy.io.fits is the type of data that is output; with it normally being an array, whereas it is a table with Astropy")


cond = (
    (data['STARFLAG'] == 0) &            #creates mask to filter the data with conditions
    (data['ASPCAPFLAG'] == 0) &
    (data['SNR'] > 10) &
    (data['GAIAEDR3_PARALLAX'] / data['GAIAEDR3_PARALLAX_ERROR'] > 5) &
    (~np.isnan(data['GAIAEDR3_PHOT_G_MEAN_MAG'])) &
    (~np.isnan(data['GAIAEDR3_PHOT_BP_MEAN_MAG'])) &
    (~np.isnan(data['GAIAEDR3_PHOT_RP_MEAN_MAG']))
)

df = cond.astype(int)

dfcolumn = fits.Column(name='DATAFLAG', array=cond, format='I')

par = data['GAIAEDR3_PARALLAX']
mg = data['GAIAEDR3_PHOT_G_MEAN_MAG']
mgl = np.full(len(data), -1)

mgl[cond] = (mg[cond])+(5*np.log10(par[cond]/1000))+5


print(df)

absm1 = np.where(df == 1, -1, mgl)
abcolumn = fits.Column(name='Abs Mag', array=absm1, format='D')
cols = hdul[1].columns + dfcolumn + abcolumn

table1 = fits.BinTableHDU.from_columns(cols)
hdul1 = fits.HDUList([hdul[0], table1])
hdul1.writeto('APOGEE_updates.fits', overwrite=True)

hdul2 = fits.open('APOGEE_updates.fits')
hdul2.info()
data2 = hdul2[1].data
print(data2.columns.names)

The difference between reading an image vs reading a table using Astropy.io.fits is the type of data that is output; with it normally being an array, whereas it is a table with Astropy
[0 0 0 ... 0 0 0]
Filename: APOGEE_updates.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      16   (847,)   uint8   
  1                1 BinTableHDU     28   733901R x 10C   [E, E, E, E, E, K, K, E, I, D]   
['GAIAEDR3_PARALLAX', 'GAIAEDR3_PARALLAX_ERROR', 'GAIAEDR3_PHOT_G_MEAN_MAG', 'GAIAEDR3_PHOT_BP_MEAN_MAG', 'GAIAEDR3_PHOT_RP_MEAN_MAG', 'ASPCAPFLAG', 'STARFLAG', 'SNR', 'DATAFLAG', 'Abs Mag']


In [8]:
#6.7 (⁶🤷⁷)
import pandas as pd 
import numpy as np 
gd = pd.read_csv('gaiaTestData.csv')
gtd = pd.DataFrame(gd)

gmf = gtd['phot_g_mean_flux']
maxgmf = gmf.max()
print(f"Max phot_g_mean_flux = {maxgmf}")

maxcol = gtd[gmf == maxgmf]
print(f"The row with the max g_mean_flux is {maxcol}")

Max phot_g_mean_flux = 42489404.0
The row with the max g_mean_flux is     source_id         ra  ra_error       dec  dec_error  phot_g_mean_flux  \
64   17418337  235.84286  0.005008  11.71101    0.00445        42489404.0   

    phot_g_mean_flux_error  phot_g_mean_mag  phot_bp_mean_flux  \
64                5649.703         6.617699         15722506.0   

    phot_bp_mean_flux_error  phot_bp_mean_mag  phot_rp_mean_flux  \
64                 9947.167          7.360095         31655918.0   

    phot_rp_mean_flux_error  phot_rp_mean_mag  a_g_val  e_bp_min_rp_val  
64                20056.184          6.010764      0.0              0.0  
